# LexisNexis Fraud Detection Project

Break Through Tech AI Studio - Fall 2026

This notebook contains exploratory analysis and modeling for detecting fraudulent insurance claims.

In [ ]:
!git clone https://github.com/Break-Through-Tech/LexisNexis-1A-detecting-fraudulent-insurance-claims.git

Cloning into 'LexisNexis-1A-detecting-fraudulent-insurance-claims'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 68 (delta 36), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 1006.41 KiB | 3.75 MiB/s, done.
Resolving deltas: 100% (36/36), done.


In [ ]:
%cd /content/LexisNexis-1A-detecting-fraudulent-insurance-claims

/content/LexisNexis-1A-detecting-fraudulent-insurance-claims


In [ ]:
!ls

Challenge-Project-Overview.md			     notebooks
data						     README.md
Getting-Started-for-Fellows.md			     requirements.txt
LexisNexis-1A-detecting-fraudulent-insurance-claims


In [ ]:
!pip install -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd



In [ ]:
df = pd.read_csv("data/fraud_oracle.csv")

In [ ]:
df.columns.tolist()

['Month',
 'WeekOfMonth',
 'DayOfWeek',
 'Make',
 'AccidentArea',
 'DayOfWeekClaimed',
 'MonthClaimed',
 'WeekOfMonthClaimed',
 'Sex',
 'MaritalStatus',
 'Age',
 'Fault',
 'PolicyType',
 'VehicleCategory',
 'VehiclePrice',
 'FraudFound_P',
 'PolicyNumber',
 'RepNumber',
 'Deductible',
 'DriverRating',
 'Days_Policy_Accident',
 'Days_Policy_Claim',
 'PastNumberOfClaims',
 'AgeOfVehicle',
 'AgeOfPolicyHolder',
 'PoliceReportFiled',
 'WitnessPresent',
 'AgentType',
 'NumberOfSuppliments',
 'AddressChange_Claim',
 'NumberOfCars',
 'Year',
 'BasePolicy']

## Task 3: Feature Leakage Audit

The purpose of this audit is to identify features that may contain
information that would not be available when an insurance claim is
initially filed.

Our intended prediction point is the time of initial claim submission.
Therefore, features generated or updated after claim submission could
introduce data leakage and artificially inflate model performance.

We also review identifier and temporal variables that may allow the model
to learn shortcuts rather than generalizable fraud patterns.

In [ ]:
print("Number of claims:", len(df))
print("Fraud cases:", df["FraudFound_P"].sum())
print("Overall fraud rate:", round(df["FraudFound_P"].mean() * 100, 2), "%")

year_summary = (
    df.groupby("Year")["FraudFound_P"]
      .agg(Claims="count",
           FraudCases="sum",
           FraudRate="mean")
)

year_summary["FraudRate"] *= 100

year_summary

Number of claims: 15420
Fraud cases: 923
Overall fraud rate: 5.99 %


,Claims,FraudCases,FraudRate
Year,,,
1994,6142,409,6.659069
1995,5195,301,5.794033
1996,4083,213,5.216752


In [ ]:
print("Number of rows:", len(df))
print("Unique PolicyNumbers:", df["PolicyNumber"].nunique())
print(
    "Percent unique:",
    round(df["PolicyNumber"].nunique() / len(df) * 100, 2),
    "%"
)

df.groupby("Year")["PolicyNumber"].agg(
    ["min", "max", "mean", "median"]
)

Number of rows: 15420
Unique PolicyNumbers: 15420
Percent unique: 100.0 %


,min,max,mean,median
Year,,,,
1994,1,6142,3071.5,3071.5
1995,6143,11337,8740.0,8740.0
1996,11338,15420,13379.0,13379.0


In [ ]:
policy_year_corr = df["PolicyNumber"].corr(df["Year"])

print(
    "Correlation between PolicyNumber and Year:",
    round(policy_year_corr, 3)
)

Correlation between PolicyNumber and Year: 0.937


### PolicyNumber decision: DROP

PolicyNumber is unique for every observation and is sequential across
the dataset. It also nearly perfectly encodes time: PolicyNumbers
1–6,142 correspond to 1994, 6,143–11,337 to 1995, and
11,338–15,420 to 1996.

Because fraud prevalence changes across years, PolicyNumber could allow
the model to learn a temporal shortcut rather than a meaningful fraud
relationship. It will therefore be excluded from model predictors.

In [ ]:
def fraud_summary(data, column):
    summary = (
        data.groupby(column, dropna=False)["FraudFound_P"]
        .agg(
            Claims="count",
            FraudCases="sum",
            FraudRate="mean"
        )
        .reset_index()
    )

    summary["FraudRate"] = (
        summary["FraudRate"] * 100
    ).round(2)

    return summary.sort_values(
        "FraudRate",
        ascending=False
    )

In [ ]:
leakage_candidates = [
    "Fault",
    "DriverRating",
    "PoliceReportFiled",
    "WitnessPresent",
    "NumberOfSuppliments",
    "AddressChange_Claim",
    "RepNumber"
]

for col in leakage_candidates:
    print(f"\n{'=' * 50}")
    print(col)
    print(f"{'=' * 50}")
    display(fraud_summary(df, col))


Fault


,Fault,Claims,FraudCases,FraudRate
0,Policy Holder,11230,886,7.89
1,Third Party,4190,37,0.88



DriverRating


,DriverRating,Claims,FraudCases,FraudRate
2,3,3884,242,6.23
3,4,3791,235,6.20
0,1,3944,232,5.88
1,2,3801,214,5.63



PoliceReportFiled


...

,NumberOfSuppliments,Claims,FraudCases,FraudRate
3,none,7047,472,6.70
0,1 to 2,2489,159,6.39
2,more than 5,3867,195,5.04
1,3 to 5,2017,97,4.81



AddressChange_Claim


,AddressChange_Claim,Claims,FraudCases,FraudRate
4,under 6 months,4,3,75.00
1,2 to 3 years,291,51,17.53
0,1 year,170,11,6.47
3,no change,14324,825,5.76
2,4 to 8 years,631,33,5.23



RepNumber


,RepNumber,Claims,FraudCases,FraudRate
5,6,942,66,7.01
6,7,1069,74,6.92
9,10,986,66,6.69
8,9,999,65,6.51
12,13,892,58,6.50
0,1,987,63,6.38
2,3,949,60,6.32
13,14,941,57,6.06
10,11,948,56,5.91
1,2,956,55,5.75


In [ ]:
fraud_summary(df, "Fault")

,Fault,Claims,FraudCases,FraudRate
0,Policy Holder,11230,886,7.89
1,Third Party,4190,37,0.88


### Fault: REVIEW

Fraud prevalence differs substantially by Fault:

- Policy Holder: approximately 7.89% fraud
- Third Party: approximately 0.88% fraud

This makes Fault potentially valuable for prediction, but predictive
strength alone does not establish leakage.

The key question is when this variable is determined. If Fault represents
information reported when the claim is initially filed, it may be a valid
predictor. If it represents a later determination made during claim
investigation, it would contain post-filing information and should be
excluded.

**Decision: REVIEW / confirm business definition before modeling.**

In [ ]:
fraud_summary(df, "NumberOfSuppliments")

,NumberOfSuppliments,Claims,FraudCases,FraudRate
3,none,7047,472,6.70
0,1 to 2,2489,159,6.39
2,more than 5,3867,195,5.04
1,3 to 5,2017,97,4.81


### NumberOfSuppliments: POTENTIAL LEAKAGE

NumberOfSuppliments records supplemental information associated with a
claim. Although its fraud rates are not dramatically different across
categories, the primary concern is timing rather than predictive strength.

If supplements accumulate after the initial claim submission, the final
number of supplements would not have been known at the intended prediction
time.

**Preliminary decision: DROP / confirm with business definition.**

In [ ]:
leakage_audit = pd.DataFrame({
    "Feature": [
        "FraudFound_P",
        "PolicyNumber",
        "Year",
        "RepNumber",
        "NumberOfSuppliments",
        "Fault",
        "DriverRating",
        "PoliceReportFiled",
        "WitnessPresent",
        "AddressChange_Claim"
    ],

    "Decision": [
        "TARGET ONLY",
        "DROP",
        "SPLIT ONLY",
        "DROP / REVIEW",
        "DROP / REVIEW",
        "KEEP / REVIEW",
        "KEEP / VERIFY",
        "KEEP / VERIFY",
        "KEEP / VERIFY",
        "KEEP / VERIFY"
    ],

    "Reason": [
        "Outcome being predicted; cannot be included as an input feature.",

        "100% unique identifier and strongly encodes time. PolicyNumber "
        "ranges perfectly separate 1994, 1995, and 1996.",

        "Needed for chronological train/validation/test splitting, but "
        "should not be used as a model predictor.",

        "Claim representative identifier. May reflect operational "
        "assignment rather than underlying claim characteristics and "
        "may not be available at the intended prediction point.",

        "Supplements may accumulate after the initial claim is filed, "
        "creating potential post-filing leakage.",

        "Strongly associated with fraud, but timing is unclear. Must "
        "confirm whether fault is reported at filing or determined later.",

        "Appears suitable if this is a pre-existing driver rating. "
        "Confirm that it was calculated before the current claim.",

        "Likely available at claim intake if the claimant reports whether "
        "a police report was filed. Confirm field timing.",

        "Likely available from the initial accident report, but confirm "
        "that the recorded value was known at claim intake.",

        "Appears to describe historical address-change information that "
        "should already exist at claim submission. Confirm definition."
    ]
})

leakage_audit

,Feature,Decision,Reason
0,FraudFound_P,TARGET ONLY,Outcome being predicted; cannot be included as...
1,PolicyNumber,DROP,100% unique identifier and strongly encodes ti...
2,Year,SPLIT ONLY,Needed for chronological train/validation/test...
3,RepNumber,DROP / REVIEW,Claim representative identifier. May reflect o...
4,NumberOfSuppliments,DROP / REVIEW,Supplements may accumulate after the initial c...
5,Fault,REVIEW,"Strongly associated with fraud, but timing is ..."
6,DriverRating,KEEP / VERIFY,Appears suitable if this is a pre-existing dri...
7,PoliceReportFiled,KEEP / VERIFY,Likely available at claim intake if the claima...
8,WitnessPresent,KEEP / VERIFY,Likely available from the initial accident rep...
9,AddressChange_Claim,KEEP / VERIFY,Appears to describe historical address-change ...


In [ ]:
likely_available_at_filing = [
    "Month",
    "WeekOfMonth",
    "DayOfWeek",
    "Make",
    "AccidentArea",
    "DayOfWeekClaimed",
    "MonthClaimed",
    "WeekOfMonthClaimed",
    "Sex",
    "MaritalStatus",
    "Age",
    "PolicyType",
    "VehicleCategory",
    "VehiclePrice",
    "Deductible",
    "Days_Policy_Accident",
    "Days_Policy_Claim",
    "PastNumberOfClaims",
    "AgeOfVehicle",
    "AgeOfPolicyHolder",
    "AgentType",
    "NumberOfCars",
    "BasePolicy"
]

print("Features provisionally available at filing:")
print(likely_available_at_filing)

Features provisionally available at filing:
['Month', 'WeekOfMonth', 'DayOfWeek', 'Make', 'AccidentArea', 'DayOfWeekClaimed', 'MonthClaimed', 'WeekOfMonthClaimed', 'Sex', 'MaritalStatus', 'Age', 'PolicyType', 'VehicleCategory', 'VehiclePrice', 'Deductible', 'Days_Policy_Accident', 'Days_Policy_Claim', 'PastNumberOfClaims', 'AgeOfVehicle', 'AgeOfPolicyHolder', 'AgentType', 'NumberOfCars', 'BasePolicy']


### Leakage Audit Conclusion

The audit identified PolicyNumber as a clear variable to exclude.
PolicyNumber is a unique sequential identifier that almost perfectly
encodes claim year, which could allow the model to exploit temporal
differences in fraud prevalence.

Year will be retained only for creation of the chronological
train/validation/test split and excluded from model predictors.

NumberOfSuppliments is a potential source of post-claim leakage and should
be excluded unless it can be confirmed that the recorded value represents
information available at initial claim submission.

Fault requires additional clarification because it has a strong
relationship with fraud but may represent either information reported at
claim intake or a determination made later in the claims process.

PoliceReportFiled, WitnessPresent, DriverRating, and AddressChange_Claim
appear potentially usable but require confirmation that their recorded
values were available at the intended prediction point.

RepNumber should also be considered for exclusion because it is an
operational identifier rather than an underlying characteristic of the
claim.